# RL Masterclass 04: Policy Gradient Methods

**Track:** Reinforcement Learning (0 to Masterclass)  
**Prerequisites:** RL 03 (DQN), 10b (PyTorch)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"DQN learns Q-values (how good is each action?), then picks the best one. Policy Gradient learns the POLICY directly (the probability of each action). Why would you learn the policy directly when you can just learn values and pick the best?"*

---

## The Paradigm Shift: Values → Policies

| | Value-Based (DQN) | Policy-Based (PG) |
|---|---|---|
| Learns | Q(s,a) → pick argmax | π(a|s) directly |
| Action space | Discrete only | Discrete AND continuous |
| Output | Deterministic (argmax) | Stochastic (probability distribution) |
| Convergence | Can oscillate | Smoother but higher variance |
| Used in | Atari, simple control | Robotics, **RLHF/PPO**, continuous control |

**Why this matters for you:** PPO (RL/05) and RLHF (RL/06, NB12) are policy gradient methods. You CANNOT understand how ChatGPT was trained without understanding this notebook.

## Table of Contents
1. The Policy Gradient Theorem
2. REINFORCE Algorithm
3. Variance Reduction: Baselines
4. Actor-Critic: Combining Values + Policy

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Policy gradient methods are the backbone of modern RL. PPO (a PG method) trains the RL step of RLHF. Understanding the policy gradient theorem explains exactly HOW a reward signal teaches an LLM to be helpful — the math is identical.

**Mental Model:** Imagine learning to play darts. Value-based is like calculating the exact score of every possible throw angle, then choosing the best one. Policy-based is like throwing darts, seeing where they land, and gradually adjusting your throwing motion. You don't need to evaluate every angle — you directly improve your technique from experience.

**Common Junior Pitfall:** Not using a baseline. Raw REINFORCE has enormous variance — the learning curve bounces wildly. Adding a baseline (subtracting the average return) dramatically reduces variance without introducing bias.

---

In [ ]:
!pip install -q torch numpy matplotlib

## 1 · The Policy Gradient Theorem

### The Core Idea

Instead of learning values, parameterize the POLICY directly:

$$\pi_\theta(a|s) = \text{probability of taking action } a \text{ in state } s$$

For a neural network: input is state $s$, output is a probability distribution over actions (via softmax).

### The Objective

Maximize the expected total reward:

$$J(\theta) = \mathbb{E}_{\pi_\theta} \left[ \sum_{t=0}^T r_t \right]$$

### The Gradient (How to Improve)

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]}$$

In plain English:
- $\log \pi_\theta(a_t|s_t)$: the log-probability of the action we took
- $G_t$: the return (how good was the outcome?)
- If $G_t$ is HIGH: increase the probability of that action (it worked!)
- If $G_t$ is LOW: decrease the probability (it failed!)

**This is EXACTLY how RLHF works**: the "return" is the reward model's score. If the LLM's response gets a high score → increase the probability of that response. Low score → decrease it.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

class PolicyNetwork(nn.Module):
    """Simple policy network: state → action probabilities.
    
    This is a STOCHASTIC policy — it outputs a probability distribution,
    not a single action. We SAMPLE actions from this distribution.
    
    In RLHF: the LLM IS the policy network.
    State = prompt + tokens generated so far
    Action = next token to generate
    π(a|s) = probability of generating token a given context s
    """
    
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
    
    def forward(self, state):
        logits = self.net(state)
        return torch.softmax(logits, dim=-1)  # Action probabilities
    
    def act(self, state):
        """Sample an action from the policy distribution."""
        state_t = torch.FloatTensor(state).unsqueeze(0)
        probs = self.forward(state_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

# Demonstrate
policy = PolicyNetwork(state_dim=4, action_dim=2)
dummy_state = np.random.randn(4)
action, log_prob = policy.act(dummy_state)
probs = policy(torch.FloatTensor(dummy_state).unsqueeze(0))
print(f'Policy Network Demo:')
print(f'  State: {dummy_state.round(3)}')
print(f'  Action probabilities: {probs.detach().numpy().round(3)}')
print(f'  Sampled action: {action}')
print(f'  Log probability: {log_prob.item():.4f}')
print(f'\n  Note: action is SAMPLED, not deterministic.')
print(f'  This stochasticity is what enables exploration in PG methods.')

---
## 2 · REINFORCE Algorithm

In [ ]:
class SimpleBalanceEnv:
    """Same environment from RL/03."""
    def __init__(self):
        self.reset()
    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, size=4)
        self.steps = 0
        return self.state.copy()
    def step(self, action):
        force = 1.0 if action == 1 else -1.0
        self.state[3] += force * 0.02 + np.random.normal(0, 0.01)
        self.state[2] += self.state[3] * 0.02
        self.state[1] += force * 0.01
        self.state[0] += self.state[1] * 0.02
        self.steps += 1
        done = abs(self.state[2]) > 0.5 or abs(self.state[0]) > 2.0 or self.steps >= 200
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

def reinforce(env, episodes=500, gamma=0.99, lr=1e-3, use_baseline=True):
    """REINFORCE (Monte Carlo Policy Gradient).
    
    Algorithm:
    1. Run a full episode, collecting (state, action, reward)
    2. Compute returns G_t for each step
    3. Compute loss = -Σ log π(a|s) × (G_t - baseline)
    4. Backpropagate and update policy
    
    The baseline reduces variance without changing the gradient's
    expected value (it's still an unbiased estimator).
    """
    policy = PolicyNetwork(state_dim=4, action_dim=2)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    all_rewards = []
    baseline = 0  # Running average of returns
    
    for episode in range(episodes):
        # --- Collect an episode ---
        state = env.reset()
        log_probs = []
        rewards = []
        
        while True:
            action, log_prob = policy.act(state)
            next_state, reward, done = env.step(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
            if done:
                break
        
        # --- Compute discounted returns ---
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.FloatTensor(returns)
        
        # --- Baseline: subtract mean return to reduce variance ---
        if use_baseline:
            baseline = 0.9 * baseline + 0.1 * returns.mean().item()
            advantages = returns - baseline
        else:
            advantages = returns
        
        # Normalize advantages (stabilizes training)
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # --- Policy gradient update ---
        # Loss = -Σ log π(a|s) × advantage
        # Negative because we MAXIMIZE reward (gradient ASCENT)
        loss = 0
        for log_prob, adv in zip(log_probs, advantages):
            loss += -log_prob * adv
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        all_rewards.append(sum(rewards))
    
    return policy, all_rewards

# Train with and without baseline
env = SimpleBalanceEnv()
print('Training REINFORCE...')
policy_bl, rewards_bl = reinforce(env, episodes=500, use_baseline=True)
print('Training REINFORCE (no baseline)...')
policy_no_bl, rewards_no_bl = reinforce(env, episodes=500, use_baseline=False)

# Compare
fig, ax = plt.subplots(figsize=(10, 5))
window = 30
smoothed_bl = [np.mean(rewards_bl[max(0,i-window):i+1]) for i in range(len(rewards_bl))]
smoothed_no = [np.mean(rewards_no_bl[max(0,i-window):i+1]) for i in range(len(rewards_no_bl))]
ax.plot(smoothed_bl, label='With Baseline', lw=2, color='steelblue')
ax.plot(smoothed_no, label='No Baseline', lw=2, color='coral', alpha=0.7)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward (smoothed)')
ax.set_title('REINFORCE: Baseline Reduces Variance')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Baseline version is more stable — less variance in the learning curve.')
print(f'This is why ALL modern PG methods use advantage estimation (GAE in PPO).')

---
## 3 · Actor-Critic: Best of Both Worlds

REINFORCE waits until the episode ends to compute returns (Monte Carlo). Actor-Critic uses a VALUE network to estimate returns at EVERY step (TD learning).

```
ACTOR (Policy Network):  π_θ(a|s) → probabilities → actions
CRITIC (Value Network):  V_φ(s)   → estimated return → tells actor how good its actions are
```

### The Advantage Function

$$A(s_t, a_t) = \underbrace{r_t + \gamma V(s_{t+1})}_{\text{what actually happened}} - \underbrace{V(s_t)}_{\text{what was expected}}$$

- $A > 0$: action was BETTER than expected → increase its probability
- $A < 0$: action was WORSE than expected → decrease its probability
- $A = 0$: exactly as expected → no change

This is the **same advantage** used in PPO (RL/05) and RLHF (RL/06).

In [ ]:
class ActorCritic(nn.Module):
    """Actor-Critic: shared backbone, separate heads.
    
    Actor:  outputs action probabilities
    Critic: outputs state value V(s)
    
    In RLHF:
      Actor  = the LLM being fine-tuned
      Critic = the value head (often added as a linear layer on top of the LLM)
    """
    
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        
        # Shared backbone
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
        )
        
        # Actor head: state → action probabilities
        self.actor = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )
        
        # Critic head: state → value
        self.critic = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    
    def forward(self, state):
        features = self.shared(state)
        action_probs = torch.softmax(self.actor(features), dim=-1)
        value = self.critic(features)
        return action_probs, value

# Architecture comparison
ac = ActorCritic(state_dim=4, action_dim=2)
state_t = torch.randn(1, 4)
probs, value = ac(state_t)

print(f'Actor-Critic Architecture:')
print(f'  Parameters: {sum(p.numel() for p in ac.parameters()):,}')
print(f'  Input:  state = {state_t.shape}')
print(f'  Actor output:  π(a|s) = {probs.detach().numpy().round(3)}')
print(f'  Critic output: V(s)   = {value.item():.3f}')
print(f'\n  Actor says: "Take action 0 with {probs[0,0]:.0%}, action 1 with {probs[0,1]:.0%}"')
print(f'  Critic says: "This state is worth {value.item():.3f}"')
print(f'\n  Advantage = (actual reward + γV(next)) - V(current)')
print(f'  If advantage > 0: action was better than expected → reinforce it')

---
## Summary: The RL Algorithm Family Tree

```
                       RL Algorithms
                      /              \
              Value-Based          Policy-Based
              /        \             /         \
         Q-Learning   DQN     REINFORCE    Actor-Critic
         (RL/02)     (RL/03)   (RL/04)      (RL/04)
                                               |
                                           A2C / A3C
                                               |
                                            ┌──PPO──┐ (RL/05)
                                            │       │
                                          TRPO    PPO-clip
                                                    |
                                                  RLHF (RL/06, NB12)
```

### 🎓 DEEP QUESTION ANSWERED

**Q:** *Why learn the policy directly instead of values?*

**A:** Three key reasons:
1. **Continuous actions**: DQN outputs Q-values for each discrete action. But what if the action is a steering angle (0°-360°)? You can't have infinite Q-values. Policy gradient naturally outputs a continuous distribution (e.g., Gaussian with learned mean and variance).
2. **Stochastic policies**: Sometimes the optimal strategy is RANDOM (rock-paper-scissors). Value-based methods are deterministic (argmax). Policy gradient naturally represents mixed strategies.
3. **RLHF**: Language model generation IS policy gradient. The LLM IS the policy. Each token is an action. The reward model scores the full response.

---

## ✅ Knowledge Check

### Q1: Log probability
Why does the policy gradient use LOG probability instead of raw probability?

<details><summary>👉 Answer</summary>

Two reasons: (1) Mathematical: the log converts the product of probabilities along a trajectory into a sum, making gradients more stable. (2) Numerical: probabilities can be extremely small (1e-30 for long sequences). Log prevents underflow. Fun fact: this is the same reason cross-entropy loss uses log probabilities in classification.
</details>

### Q2: REINFORCE vs Actor-Critic
REINFORCE waits until the episode ends. Actor-Critic updates every step. Which is better for long episodes (1000+ steps)?

<details><summary>👉 Answer</summary>

Actor-Critic is much better for long episodes. REINFORCE must store the ENTIRE episode before ANY learning happens. With 1000+ steps, the returns are noisy and learning is extremely slow. Actor-Critic uses TD learning (bootstrap from V estimate) to update at every step, providing faster, lower-variance feedback to the policy.
</details>

### Q3: Connection to RLHF
In RLHF, what is the "state", "action", "policy", and "reward"?

<details><summary>👉 Answer</summary>

State = the prompt + all tokens generated so far. Action = the next token to generate. Policy = the LLM (π_θ is the token probability distribution). Reward = the reward model's score for the complete response. The RLHF training step uses PPO to increase the probability of high-scoring token sequences.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Continuous Actions
Modify the PolicyNetwork to output a Gaussian distribution (mean + std) instead of categorical probabilities. Sample actions from this Gaussian. This is how continuous control works.

### Exercise 2: A2C
Implement Advantage Actor-Critic (A2C): train the critic to minimize MSE(V(s), R_actual), and train the actor using the advantage A = R - V(s). Compare to REINFORCE.

### Exercise 3: Entropy Bonus
Add an entropy bonus to the policy loss: loss = -log π(a|s) × A - β × H(π). This encourages exploration by penalizing overly confident policies. Experiment with different β values.

---

**Next →** RL 05: Proximal Policy Optimization (PPO)